# 📝 Type Hints, Enums, Logging, Config, Packaging
### Exercises & Solutions — 25 Problems

This notebook is exercises-and-solutions only. It assumes you've already covered the
concept notebook for this topic. Each problem targets a **distinct function, pattern,
or real-world scenario** so that working through all of them gives you practical
exposure to everything commonly used on the job.

**Coverage map:**

- Type hints: basic, Optional, Union, generics, Protocol, TypedDict, overload (1-9)
- Enums: basic, IntEnum, Flag, auto(), methods on enums (10-14)
- Logging: handlers, formatters, levels, structured logging (15-19)
- Config: env vars, dataclass settings, configparser, validation (20-23)
- Packaging: __init__.py exports, entry points (24-25)


---


### 1. Basic Function Type Hints + mypy-style self-check

Write a fully type-hinted `calculate_discount(price: float, pct: int) -> float` and manually verify behaviour with both valid and edge-case inputs.

In [ ]:
def calculate_discount(price: float, pct: int) -> float:
    return round(price * (1 - pct / 100), 2)

print(calculate_discount(100.0, 20))
print(calculate_discount(49.99, 0))
print(calculate_discount(200.0, 100))

# Type hints are NOT enforced at runtime - Python still type-checks at the
# operator level though, so wrong types often surface as a real runtime error:
try:
    calculate_discount("100", 20)   # str doesn't support this arithmetic -> real TypeError
except TypeError as e:
    print(f"Runtime error despite no static check: {e}")
print("mypy would have caught this BEFORE running, with zero runtime cost")


### 2. Optional and Union for Nullable / Multi-Type Returns

Write `parse_int(s: str) -> Optional[int]` (returns None on failure) and `format_value(v: Union[int, float, str]) -> str`.

In [ ]:
from typing import Optional, Union

def parse_int(s: str) -> Optional[int]:
    try:
        return int(s)
    except ValueError:
        return None

def format_value(v: Union[int, float, str]) -> str:
    if isinstance(v, float):
        return f"{v:.2f}"
    return str(v)

print(parse_int("42"), parse_int("abc"))
print(format_value(3.14159), format_value(42), format_value("text"))

### 3. List, Dict, Tuple Generic Hints

Write a fully-hinted `group_by_length(words: List[str]) -> Dict[int, List[str]]` grouping words by length.

In [ ]:
from typing import List, Dict

def group_by_length(words: List[str]) -> Dict[int, List[str]]:
    groups: Dict[int, List[str]] = {}
    for w in words:
        groups.setdefault(len(w), []).append(w)
    return groups

print(group_by_length(["cat", "dog", "bird", "ox", "fish"]))

### 4. Callable Type Hints for Higher-Order Functions

Write `apply_n_times(fn: Callable[[int], int], x: int, n: int) -> int` that applies `fn` to `x` repeatedly `n` times, fully type-hinted.

In [ ]:
from typing import Callable

def apply_n_times(fn: Callable[[int], int], x: int, n: int) -> int:
    for _ in range(n):
        x = fn(x)
    return x

print(apply_n_times(lambda x: x * 2, 1, 5))    # 1*2*2*2*2*2 = 32
print(apply_n_times(lambda x: x + 10, 0, 3))   # 30

### 5. Generic Class with TypeVar

Build a generic `Box[T]` class that can hold and retrieve a typed value, demonstrating type-safe generics usable for ANY type.

In [ ]:
from typing import TypeVar, Generic

T = TypeVar("T")

class Box(Generic[T]):
    def __init__(self, item: T) -> None:
        self._item = item
    def get(self) -> T:
        return self._item
    def set(self, item: T) -> None:
        self._item = item

int_box: Box[int] = Box(42)
str_box: Box[str] = Box("hello")
print(int_box.get(), str_box.get())
int_box.set(100)
print(int_box.get())

### 6. Bounded TypeVar for Constrained Generics

Build a generic `find_max(items: List[T]) -> T` constrained so `T` must support comparison (`TypeVar` with bound on a Protocol).

In [ ]:
from typing import TypeVar, List, Protocol

class Comparable(Protocol):
    def __lt__(self, other) -> bool: ...

CT = TypeVar("CT", bound=Comparable)

def find_max(items: List[CT]) -> CT:
    result = items[0]
    for item in items[1:]:
        if result < item:
            result = item
    return result

print(find_max([3, 7, 2, 9, 4]))
print(find_max(["banana", "apple", "cherry"]))

### 7. Protocol for Structural Typing

Build a `Flushable` Protocol with a `flush()` method, and a function `flush_all(items)` that works on ANY objects satisfying the protocol — no inheritance required.

In [ ]:
from typing import Protocol, List

class Flushable(Protocol):
    def flush(self) -> None: ...

class Buffer:
    def __init__(self): self.flushed = False
    def flush(self) -> None: self.flushed = True

class Logger:
    def __init__(self): self.flushed = False
    def flush(self) -> None: self.flushed = True

def flush_all(items: List[Flushable]) -> None:
    for item in items:
        item.flush()

items = [Buffer(), Logger()]
flush_all(items)
print([item.flushed for item in items])

### 8. TypedDict for Structured Dict Validation

Use `TypedDict` to define a `UserRecord` shape (`id: int, name: str, active: bool`) and write a function consuming it with full type safety.

In [ ]:
from typing import TypedDict

class UserRecord(TypedDict):
    id: int
    name: str
    active: bool

def describe_user(user: UserRecord) -> str:
    status = "active" if user["active"] else "inactive"
    return f"User #{user['id']}: {user['name']} ({status})"

u: UserRecord = {"id": 1, "name": "Alice", "active": True}
print(describe_user(u))

### 9. @overload for Multiple Type Signatures

Use `typing.overload` to declare that `process(x: int) -> int` and `process(x: str) -> str` have different (but type-consistent) signatures for static checkers.

In [ ]:
from typing import overload, Union

@overload
def process(x: int) -> int: ...
@overload
def process(x: str) -> str: ...

def process(x: Union[int, str]) -> Union[int, str]:
    if isinstance(x, int):
        return x * 2
    return x.upper()

print(process(5))        # mypy knows this returns int
print(process("hello"))  # mypy knows this returns str

### 10. Basic Enum with Methods

Build a `Direction(Enum)` with `NORTH, SOUTH, EAST, WEST` and a method `opposite()` returning the reverse direction.

In [ ]:
from enum import Enum

class Direction(Enum):
    NORTH = "N"
    SOUTH = "S"
    EAST = "E"
    WEST = "W"

    def opposite(self):
        pairs = {
            Direction.NORTH: Direction.SOUTH,
            Direction.SOUTH: Direction.NORTH,
            Direction.EAST: Direction.WEST,
            Direction.WEST: Direction.EAST,
        }
        return pairs[self]

print(Direction.NORTH.opposite())
print(Direction.WEST.opposite())
for d in Direction:
    print(f"  {d.name} ({d.value}) <-> {d.opposite().name}")

### 11. IntEnum for Numeric Comparisons

Build a `LogLevel(IntEnum)` matching Python's real logging levels, then write a function `should_log(current, threshold)` using `>=` comparison directly on enum members.

In [ ]:
from enum import IntEnum

class LogLevel(IntEnum):
    DEBUG = 10
    INFO = 20
    WARNING = 30
    ERROR = 40
    CRITICAL = 50

def should_log(message_level: LogLevel, threshold: LogLevel) -> bool:
    return message_level >= threshold      # works because IntEnum IS an int

print(should_log(LogLevel.ERROR, LogLevel.WARNING))   # True
print(should_log(LogLevel.DEBUG, LogLevel.WARNING))   # False
print(LogLevel.ERROR > 25)                              # True - compares directly to plain int too!

### 12. Flag Enum for Bitwise Combination

Build a `FilePermission(Flag)` with `READ, WRITE, EXECUTE` and demonstrate combining/checking permissions with `|` and `in`.

In [ ]:
from enum import Flag, auto

class FilePermission(Flag):
    NONE = 0
    READ = auto()
    WRITE = auto()
    EXECUTE = auto()
    ALL = READ | WRITE | EXECUTE

user_perms = FilePermission.READ | FilePermission.WRITE
print("Combined:", user_perms)
print("Has READ:", FilePermission.READ in user_perms)
print("Has EXECUTE:", FilePermission.EXECUTE in user_perms)
print("Equals ALL:", user_perms == FilePermission.ALL)
print("ALL perms:", FilePermission.ALL)

### 13. Enum with auto() and Custom __str__

Build a `Status(Enum)` using `auto()` for values, with a custom `__str__` returning a friendly capitalized label instead of the default repr.

In [ ]:
from enum import Enum, auto

class Status(Enum):
    PENDING = auto()
    IN_PROGRESS = auto()
    COMPLETED = auto()
    CANCELLED = auto()

    def __str__(self):
        return self.name.replace("_", " ").title()

for s in Status:
    print(f"{s.name} = {s.value} -> displays as: {s}")

### 14. Enum-based State Machine with Valid Transitions

Build a `TrafficLight(Enum)` with a `next_state()` method enforcing the cycle RED -> GREEN -> YELLOW -> RED, disallowing invalid jumps.

In [ ]:
from enum import Enum

class TrafficLight(Enum):
    RED = "red"
    GREEN = "green"
    YELLOW = "yellow"

    def next_state(self):
        transitions = {
            TrafficLight.RED: TrafficLight.GREEN,
            TrafficLight.GREEN: TrafficLight.YELLOW,
            TrafficLight.YELLOW: TrafficLight.RED,
        }
        return transitions[self]

light = TrafficLight.RED
for _ in range(5):
    print(light.value, end=" -> ")
    light = light.next_state()
print(light.value)

### 15. Multiple Handlers with Different Levels

Set up a logger with TWO handlers: console (INFO+) and an in-memory `StringIO` stream (DEBUG+), proving each handler filters independently.

In [ ]:
import logging, io

logger = logging.getLogger("multi_handler_demo")
logger.setLevel(logging.DEBUG)
logger.handlers.clear()   # avoid duplicate handlers if cell re-run

console = logging.StreamHandler()
console.setLevel(logging.INFO)
logger.addHandler(console)

buffer = io.StringIO()
file_like = logging.StreamHandler(buffer)
file_like.setLevel(logging.DEBUG)
logger.addHandler(file_like)

logger.debug("Debug message")     # only in buffer, not console
logger.info("Info message")        # in BOTH

print("\n--- Captured in buffer (DEBUG+) ---")
print(buffer.getvalue())

### 16. Custom Formatter with Extra Fields

Build a logger using a custom `Formatter` string including `%(filename)s:%(lineno)d`, and use the `extra=` parameter to inject a custom `request_id` field.

In [ ]:
import logging

logger = logging.getLogger("custom_fmt_demo")
logger.setLevel(logging.INFO)
logger.handlers.clear()

handler = logging.StreamHandler()
formatter = logging.Formatter("[%(levelname)s] req=%(request_id)s %(message)s")
handler.setFormatter(formatter)
logger.addHandler(handler)

logger.info("Processing payment", extra={"request_id": "abc-123"})
logger.warning("Slow response", extra={"request_id": "xyz-789"})

### 17. Exception Logging with Traceback

Use `logger.exception()` inside an `except` block to automatically capture and log the FULL traceback, vs `logger.error()` which doesn't.

In [ ]:
import logging

logger = logging.getLogger("exc_demo")
logger.setLevel(logging.ERROR)
logger.handlers.clear()
logger.addHandler(logging.StreamHandler())

def risky():
    return 1 / 0

try:
    risky()
except ZeroDivisionError:
    logger.exception("Calculation failed")   # includes full traceback automatically

try:
    risky()
except ZeroDivisionError as e:
    logger.error(f"Calculation failed: {e}")  # message only, NO traceback

### 18. Child Loggers and Hierarchical Propagation

Build a parent logger `app` and child logger `app.database`, showing how child logger messages propagate up to the parent's handlers automatically.

In [ ]:
import logging

logging.getLogger("app").handlers.clear()
parent = logging.getLogger("app")
parent.setLevel(logging.INFO)
parent.addHandler(logging.StreamHandler())

child = logging.getLogger("app.database")    # dotted name = child of "app"
child.setLevel(logging.DEBUG)
# child has NO handler of its own - messages propagate up to parent's handler

print("Child's effective level:", logging.getLevelName(child.getEffectiveLevel()))
child.info("Connected to database")   # appears via PARENT's handler, not child's
print("Propagate flag:", child.propagate)

### 19. Structured (JSON-style) Logging

Build a custom `Formatter` subclass that outputs each log record as a JSON string, useful for log aggregation systems (e.g. ELK, Datadog).

In [ ]:
import logging, json, time

class JSONFormatter(logging.Formatter):
    def format(self, record):
        payload = {
            "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S", time.localtime(record.created)),
            "level": record.levelname,
            "logger": record.name,
            "message": record.getMessage(),
        }
        return json.dumps(payload)

logger = logging.getLogger("json_demo")
logger.setLevel(logging.INFO)
logger.handlers.clear()
handler = logging.StreamHandler()
handler.setFormatter(JSONFormatter())
logger.addHandler(handler)

logger.info("User logged in")
logger.warning("Rate limit approaching")

### 20. Typed Settings from Environment Variables

Build a `Settings` dataclass with a `from_env()` classmethod parsing/casting environment variables, including a boolean and a list (comma-separated).

In [ ]:
import os
from dataclasses import dataclass
from typing import List

@dataclass(frozen=True)
class Settings:
    debug: bool
    max_workers: int
    allowed_origins: List[str]

    @classmethod
    def from_env(cls):
        return cls(
            debug=os.environ.get("DEBUG", "false").lower() in ("1", "true", "yes"),
            max_workers=int(os.environ.get("MAX_WORKERS", "4")),
            allowed_origins=os.environ.get("ALLOWED_ORIGINS", "localhost").split(","),
        )

os.environ["DEBUG"] = "true"
os.environ["MAX_WORKERS"] = "8"
os.environ["ALLOWED_ORIGINS"] = "api.example.com,app.example.com"

settings = Settings.from_env()
print(settings)

### 21. Config Validation Raising Clear Errors

Write a `validate_config(config: dict) -> None` that checks required keys exist AND have correct types, collecting ALL errors before raising (not just the first).

In [ ]:
class ConfigError(Exception):
    pass

SCHEMA = {"host": str, "port": int, "debug": bool}

def validate_config(config: dict) -> None:
    errors = []
    for key, expected_type in SCHEMA.items():
        if key not in config:
            errors.append(f"missing required key: '{key}'")
        elif not isinstance(config[key], expected_type):
            errors.append(f"'{key}' must be {expected_type.__name__}, got {type(config[key]).__name__}")
    if errors:
        raise ConfigError("; ".join(errors))

try:
    validate_config({"host": "localhost", "port": "8080"})   # port is wrong type, debug missing
except ConfigError as e:
    print("Validation failed:", e)

validate_config({"host": "localhost", "port": 8080, "debug": True})
print("Valid config passed")

### 22. configparser for INI-style Config with Sections

Use `configparser` to read a multi-section INI config, including type coercion helpers (`getint`, `getboolean`) and a fallback default.

In [ ]:
import configparser

config = configparser.ConfigParser()
config.read_string("""
[server]
host = 0.0.0.0
port = 8080
debug = yes

[database]
url = postgresql://localhost/mydb
pool_size = 10
""")

print("Host:", config.get("server", "host"))
print("Port:", config.getint("server", "port"))
print("Debug:", config.getboolean("server", "debug"))
print("Pool size:", config.getint("database", "pool_size"))
print("Missing key with fallback:", config.get("server", "timeout", fallback="30"))
print("\nAll sections:", config.sections())

### 23. Layered Config: Defaults < File < Environment Override

Build a config loader that merges THREE layers with increasing priority: hardcoded defaults, a config file dict, then environment variable overrides.

In [ ]:
import os

DEFAULTS = {"timeout": 30, "retries": 3, "log_level": "INFO"}

def load_layered_config(file_config: dict) -> dict:
    config = {**DEFAULTS, **file_config}    # file overrides defaults
    # Environment variables override everything (highest priority)
    for key in config:
        env_key = key.upper()
        if env_key in os.environ:
            raw = os.environ[env_key]
            # preserve original type
            original_type = type(config[key])
            config[key] = original_type(raw) if original_type != bool else raw.lower() == "true"
    return config

file_cfg = {"timeout": 60, "retries": 5}
os.environ["RETRIES"] = "10"      # this should win over both defaults AND file config

final = load_layered_config(file_cfg)
print(final)
print("Priority proven: retries =", final["retries"], "(from env, not file's 5 or default's 3)")

### 24. Curating a Package's Public API via __init__.py

Simulate (in a single notebook cell, using exec/modules) how `__init__.py` re-exports selected names and defines `__all__` to control `from package import *`.

In [ ]:
import types

# Simulating a package's internal modules
core_module = types.ModuleType("core")
exec("def process(): return 'processed'\nclass Engine: pass", core_module.__dict__)

utils_module = types.ModuleType("utils")
exec("def helper(): return 'helped'\ndef _internal_only(): return 'hidden'", utils_module.__dict__)

# Simulating __init__.py curating the public API
package_init = types.ModuleType("mypackage")
package_init.process = core_module.process
package_init.Engine = core_module.Engine
package_init.helper = utils_module.helper
package_init.__all__ = ["process", "Engine", "helper"]   # _internal_only is NOT exposed

print("Public API:", package_init.__all__)
print(package_init.process())
print(package_init.helper())
print("'_internal_only' exposed at package level:", hasattr(package_init, "_internal_only"))

### 25. pyproject.toml Entry Points — Parsing & Validating Structure

Write a function that parses a `pyproject.toml`-style dict structure and validates it has the required `[project]` fields before a (simulated) build/publish step.

In [ ]:
def validate_pyproject(data: dict) -> list:
    errors = []
    project = data.get("project", {})
    for required in ["name", "version"]:
        if required not in project:
            errors.append(f"[project] missing required field: '{required}'")
    deps = project.get("dependencies", [])
    if not isinstance(deps, list):
        errors.append("[project.dependencies] must be a list")
    scripts = data.get("project", {}).get("scripts", {})
    for cli_name, target in scripts.items():
        if ":" not in target:
            errors.append(f"script '{cli_name}' target '{target}' must be 'module:function' format")
    return errors

good_config = {
    "project": {
        "name": "my-package", "version": "0.1.0",
        "dependencies": ["requests>=2.28"],
        "scripts": {"my-cli": "my_package.cli:main"}
    }
}
bad_config = {
    "project": {
        "dependencies": "not-a-list",
        "scripts": {"my-cli": "broken_target_no_colon"}
    }
}

print("Good config errors:", validate_pyproject(good_config))
print("Bad config errors:", validate_pyproject(bad_config))